# OGENTI Training - Llama 4 Scout on A100 80GB

**O Series Protocol Training Pipeline**

| Spec | Value |
|------|-------|
| Model | Llama 4 Scout 17B-16E (109B MoE) |
| GPU | NVIDIA A100 80GB |
| Method | QLoRA (4-bit NF4 + LoRA r=16) |
| Framework | MAPPO (Multi-Agent PPO) |
| Phases | 5-phase curriculum (1200 episodes) |

---

### Prerequisites
- Google Colab Pro/Pro+ (A100 GPU runtime)
- HuggingFace account with Llama 4 access approved
- GitHub repo access (if private)

## 0. GPU Check

In [ ]:
import torch

print("=" * 60)
print("GPU STATUS CHECK")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
    print(f"PyTorch: {torch.__version__}")
    if gpu_mem < 70:
        print(f"\nWARNING: Got {gpu_mem:.0f}GB VRAM. Scout 4-bit needs ~60GB.")
        print("Consider switching to A100 80GB runtime.")
    else:
        print(f"\nPERFECT. {gpu_mem:.0f}GB is more than enough!")
else:
    print("NO GPU DETECTED!")
    print("Go to: Runtime > Change runtime type > GPU (A100)")
    raise RuntimeError("GPU required")

print("\n" + "=" * 60)
!nvidia-smi

## 1. Install Dependencies

In [ ]:
%%time
!pip install -q --upgrade \
    torch \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    peft>=0.13.0 \
    trl>=0.12.0 \
    datasets \
    bitsandbytes>=0.43.0 \
    scipy \
    sentencepiece \
    protobuf

# Web dashboard
!pip install -q fastapi uvicorn websockets

# Verification
import transformers, peft, trl, bitsandbytes, accelerate
print(f"transformers: {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"trl:           {trl.__version__}")
print(f"bitsandbytes:  {bitsandbytes.__version__}")
print(f"accelerate:    {accelerate.__version__}")
print("\nAll dependencies installed!")

## 2. HuggingFace Login

Llama 4 Scout is a **gated model**. You need:
1. Go to https://huggingface.co/meta-llama/Llama-4-Scout-17B-16E-Instruct
2. Accept the license agreement
3. Create a token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

# Paste your HuggingFace token below
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

login(token=HF_TOKEN)
print("Logged in!")

## 3. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/gkjuwon-ui/ai-master.git"
REPO_DIR = "/content/ai_master"

if os.path.exists(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
!ls -la

## 4. Verify Config and Model Access

In [ ]:
import json

CONFIG_PATH = "configs/a100_scout.json"

with open(CONFIG_PATH) as f:
    config = json.load(f)

model_name = config["encoder"]["model_name"]
quant = config["encoder"]["quantization"]
total_ep = config["total_episodes"]
phases = config["phases"]

print("=" * 60)
print("TRAINING CONFIG SUMMARY")
print("=" * 60)
print(f"Model:           {model_name}")
print(f"Quantization:    {quant}")
print(f"LoRA rank:       {config['encoder']['lora_rank']}")
print(f"LoRA alpha:      {config['encoder']['lora_alpha']}")
print(f"Total episodes:  {total_ep}")
print(f"Phases:          {len(phases)}")
for p in phases:
    print(f"  Phase {p['phase_id']}: {p['name']:15s} ({p['max_episodes']} ep, lr={p['learning_rate']})")
print(f"Eval every:      {config['eval_every']}")
print(f"Checkpoint dir:  {config['infra']['checkpoint_dir']}")
print("=" * 60)

# Test model access
print(f"\nTesting access to {model_name}...")
from transformers import AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Model accessible! Vocab size: {len(tok):,}")
    del tok
except Exception as e:
    print(f"Cannot access model: {e}")
    print("Make sure you accepted the Llama 4 license on HuggingFace.")

## 5. Memory Estimation

In [ ]:
import torch

gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

model_params_b = 109  # billion
weight_mem_gb = model_params_b * 0.5  # 4-bit = 0.5 bytes/param

lora_rank = config["encoder"]["lora_rank"]
lora_params = lora_rank * 4096 * 2 * 48 * 4 * 2
lora_mem_gb = lora_params * 4 / 1e9
optimizer_mem_gb = lora_params * 8 / 1e9
activation_gb = 3.0

total_estimated = weight_mem_gb + lora_mem_gb + optimizer_mem_gb + activation_gb

print("=" * 60)
print("VRAM ESTIMATION (approximate)")
print("=" * 60)
print(f"Model weights (4-bit):    {weight_mem_gb:.1f} GB")
print(f"LoRA trainable params:    {lora_mem_gb:.1f} GB")
print(f"Optimizer states:         {optimizer_mem_gb:.1f} GB")
print(f"Activations (grad ckpt):  {activation_gb:.1f} GB")
print("-" * 40)
print(f"Estimated total:          {total_estimated:.1f} GB")
print(f"Available VRAM:           {gpu_mem:.1f} GB")
print(f"Headroom:                 {gpu_mem - total_estimated:.1f} GB")
print("=" * 60)

if total_estimated > gpu_mem * 0.95:
    print("\nTIGHT FIT! Consider reducing batch_size to 2 or 1.")
else:
    print(f"\nLOOKS GOOD! {gpu_mem - total_estimated:.1f}GB headroom.")

## 6. Launch Training

Main training cell. Expected runtime:
- 1200 episodes on A100 80GB
- ~2-4 hours depending on episode complexity

Checkpoints saved every 100 episodes.

In [ ]:
import subprocess, sys, os

os.chdir("/content/ai_master")
os.environ["GRADIENT_CHECKPOINTING"] = "1"

cmd = [
    sys.executable, "run_production.py",
    "--config", "configs/a100_scout.json",
    "--no-wandb",
    "--port", "8000",
]

print("Starting OGENTI training...")
print(f"Command: {' '.join(cmd)}")
print("=" * 60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
    process.wait()
    print("\n" + "=" * 60)
    print(f"Training finished with exit code: {process.returncode}")
except KeyboardInterrupt:
    print("\nTraining interrupted! Checkpoints are saved.")
    process.terminate()

## 7. Resume Training (if interrupted)

If the runtime disconnected or you stopped training, resume from the last checkpoint:

In [ ]:
import subprocess, sys, os

os.chdir("/content/ai_master")
CHECKPOINT_DIR = "checkpoints/a100_scout"

if os.path.exists(CHECKPOINT_DIR):
    files = os.listdir(CHECKPOINT_DIR)
    print(f"Found {len(files)} checkpoint files:")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("No checkpoints found. Run training first.")
    raise FileNotFoundError(CHECKPOINT_DIR)

print(f"\nResuming from {CHECKPOINT_DIR}...")

cmd = [
    sys.executable, "run_production.py",
    "--config", "configs/a100_scout.json",
    "--no-wandb",
    "--port", "8000",
    "--resume", CHECKPOINT_DIR,
]

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
    process.wait()
    print(f"\nResumed training finished with exit code: {process.returncode}")
except KeyboardInterrupt:
    print("\nTraining interrupted!")
    process.terminate()

## 8. Monitor Training Log

In [ ]:
import os

LOG_FILE = "/content/ai_master/training.log"

if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        lines = f.readlines()
    print(f"Total log lines: {len(lines)}")
    print("\n--- Last 50 lines ---")
    for line in lines[-50:]:
        print(line, end="")
else:
    print("No training.log found yet. Start training first.")

## 9. Analyze Results

In [ ]:
import json, os, re

CKPT_DIR = "checkpoints/a100_scout"
LOG_FILE = "training.log"

episodes = []
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        for line in f:
            m = re.search(r"Ep\s+(\d+).*R=\s*([\d.]+).*acc=\s*([\d.]+).*comp=\s*([\d.]+)", line)
            if m:
                episodes.append({
                    "ep": int(m.group(1)),
                    "reward": float(m.group(2)),
                    "accuracy": float(m.group(3)),
                    "compression": float(m.group(4)),
                })

if episodes:
    print(f"Parsed {len(episodes)} episodes from log")
    print("\n" + "=" * 70)
    print(f"{'Metric':<20} {'Best':>10} {'Last 10 Avg':>12} {'Overall Avg':>12}")
    print("=" * 70)
    for metric in ["reward", "accuracy", "compression"]:
        values = [e[metric] for e in episodes]
        last_10 = values[-10:] if len(values) >= 10 else values
        best = max(values)
        avg_last = sum(last_10) / len(last_10)
        avg_all = sum(values) / len(values)
        scale = 100 if metric == "accuracy" else 1
        print(f"{metric:<20} {best*scale:>10.2f} {avg_last*scale:>11.2f} {avg_all*scale:>11.2f}")
    print("=" * 70)
    best_ep = max(episodes, key=lambda e: e["reward"])
    print(f"\nBest Episode: {best_ep['ep']}")
    print(f"   R={best_ep['reward']:.4f}, acc={best_ep['accuracy']*100:.2f}%, comp={best_ep['compression']:.1f}x")
else:
    print("No episode data found in training.log")

final_state = os.path.join(CKPT_DIR, "state_final.json")
if os.path.exists(final_state):
    with open(final_state) as f:
        state = json.load(f)
    print(f"\nFinal state: {json.dumps(state, indent=2)}")
elif os.path.exists(CKPT_DIR):
    states = sorted(
        [f for f in os.listdir(CKPT_DIR) if f.startswith("state_")],
        key=lambda x: os.path.getmtime(os.path.join(CKPT_DIR, x))
    )
    if states:
        with open(os.path.join(CKPT_DIR, states[-1])) as f:
            state = json.load(f)
        print(f"\nLatest checkpoint ({states[-1]}): {json.dumps(state, indent=2)}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not episodes:
    print("No data to plot. Run training first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    eps = [e["ep"] for e in episodes]
    window = min(20, len(episodes) // 5) or 1

    for ax, key, color, title in [
        (axes[0], "reward", "blue", "Reward"),
        (axes[1], "accuracy", "green", "Accuracy"),
        (axes[2], "compression", "red", "Compression (x)"),
    ]:
        vals = [e[key] * (100 if key == "accuracy" else 1) for e in episodes]
        ax.plot(eps, vals, alpha=0.3, color=color)
        ma = np.convolve(vals, np.ones(window)/window, mode="valid")
        ax.plot(eps[window-1:], ma, color=color, linewidth=2, label=f"MA-{window}")
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Episode")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("OGENTI Training - Llama 4 Scout QLoRA on A100", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to training_curves.png")

## Download Results

Download checkpoints, adapter weights, and training log.

In [ ]:
import os
from google.colab import files

CKPT_DIR = "checkpoints/a100_scout"
ARCHIVE = "/content/ogenti_scout_results.tar.gz"

items = []
if os.path.exists(CKPT_DIR): items.append(CKPT_DIR)
if os.path.exists("training.log"): items.append("training.log")
if os.path.exists("training_curves.png"): items.append("training_curves.png")

if items:
    !tar -czf {ARCHIVE} {" ".join(items)}
    size_mb = os.path.getsize(ARCHIVE) / 1e6
    print(f"Archive created: {ARCHIVE} ({size_mb:.1f} MB)")
    files.download(ARCHIVE)
else:
    print("No results to download. Run training first.")

## Save to Google Drive (Optional)

Mount Google Drive and save checkpoints for persistence.

In [ ]:
import os, shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ogenti_results"
os.makedirs(DRIVE_DIR, exist_ok=True)

CKPT_DIR = "/content/ai_master/checkpoints/a100_scout"
if os.path.exists(CKPT_DIR):
    dst = os.path.join(DRIVE_DIR, "a100_scout")
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(CKPT_DIR, dst)
    print(f"Checkpoints saved to {dst}")

for src in ["training.log", "training_curves.png"]:
    full = f"/content/ai_master/{src}"
    if os.path.exists(full):
        shutil.copy2(full, DRIVE_DIR)
        print(f"Saved {src}")

print(f"\nAll results saved to: {DRIVE_DIR}")

---

## Fallback: Smaller Model Options

If Llama 4 Scout doesn't fit or isn't accessible, try these alternatives:

| Model | Params | 4-bit VRAM | Notes |
|-------|--------|------------|-------|
| `meta-llama/Llama-4-Scout-17B-16E-Instruct` | 109B MoE | ~55 GB | **Target** |
| `meta-llama/Llama-3.3-70B-Instruct` | 70B | ~35 GB | Solid alternative |
| `Qwen/Qwen2.5-32B-Instruct` | 32B | ~16 GB | Fast, good quality |
| `Qwen/Qwen2.5-14B-Instruct` | 14B | ~7 GB | Fastest, fits any GPU |

To switch models, edit the config:

In [ ]:
# Quick model swap (run this BEFORE the training cell)
# Uncomment the model you want to use:

# FALLBACK_MODEL = "meta-llama/Llama-3.3-70B-Instruct"  # 35GB in 4-bit
# FALLBACK_MODEL = "Qwen/Qwen2.5-32B-Instruct"          # 16GB in 4-bit
# FALLBACK_MODEL = "Qwen/Qwen2.5-14B-Instruct"          # 7GB in 4-bit

# To apply:
# import json
# with open("configs/a100_scout.json") as f:
#     cfg = json.load(f)
# cfg["encoder"]["model_name"] = FALLBACK_MODEL
# cfg["decoder"]["model_name"] = FALLBACK_MODEL
# with open("configs/a100_scout.json", "w") as f:
#     json.dump(cfg, f, indent=2)
# print(f"Switched to {FALLBACK_MODEL}")